# 诗歌生成

# 数据处理

In [6]:
import numpy as np
import tensorflow as tf
import collections

from sqlalchemy.testing.util import random_choices
from tensorflow.python import keras as ks
from tensorflow.python.keras import layers
from tensorflow.python.keras import optimizers#, datasets


start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r',encoding='utf-8', errors='replace') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset():
    instances, word2id, id2word = process_dataset('poems.txt')
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    ds = ds.padded_batch(100, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

# 模型代码， 完成建模代码

In [7]:
class myRNNModel(ks.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)  # 词表大小
        # 词嵌入层：将离散的词ID映射为连续向量
        self.embed_layer = ks.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        # RNN单元：使用简单RNN结构，隐藏单元数128
        self.rnncell = ks.layers.SimpleRNNCell(128)
         # RNN层：包装RNN单元，自动处理序列迭代
        self.rnn_layer = ks.layers.RNN(self.rnncell, return_sequences=True)
        # 全连接层：将RNN输出映射到词表空间
        self.dense = ks.layers.Dense(self.v_sz)
        
    @tf.function
    def call(self, inp_ids):
        '''
        此处完成建模过程，可以参考Learn2Carry
        '''
        """前向传播过程（训练模式）
        参数:
            inp_ids: 输入序列张量，形状 [batch_size, seq_len]
        返回:
            logits: 预测逻辑值，形状 [batch_size, seq_len, vocab_size]
        """
        # 词嵌入层转换：[batch, seq_len] -> [batch, seq_len, 64]
        embeddings = self.embed_layer(inp_ids)
        
        # RNN处理序列：[batch, seq_len, 64] -> [batch, seq_len, 128]
        # 返回每个时间步的输出（return_sequences=True）
        rnn_output = self.rnn_layer(embeddings)
        
        # 全连接层映射到词表空间：[batch, seq_len, 128] -> [batch, seq_len, vocab_size]
        logits = self.dense(rnn_output)
        return logits
    
    @tf.function
    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,] 
        '''
        """单步预测方法（推理模式）
        参数:
            x: 当前时刻的输入token ID，形状 [batch_size]
            state: RNN的当前状态
        返回:
            out: 预测的下一个token ID，形状 [batch_size]
            new_state: 更新后的RNN状态
        """
        # 词嵌入转换：[batch] -> [batch, 64]
        inp_emb = self.embed_layer(x) #shape(b_sz, emb_sz)
        
         # RNN单元单步计算：[batch, 64] + 状态 -> [batch, 128] + 新状态
        h, state = self.rnncell.call(inp_emb, state) # shape(b_sz, h_sz)
        
         # 生成预测逻辑值：[batch, 128] -> [batch, vocab_size]
        logits = self.dense(h) # shape(b_sz, v_sz)
        
         # 选择概率最高的token ID：[batch, vocab_size] -> [batch]
        out = tf.argmax(logits, axis=-1)
        return out, state

## 一个计算sequence loss的辅助函数，只需了解用途。

In [8]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [9]:
from tensorflow.python.eager import tape


@tf.function
def compute_loss(logits, labels, seqlen):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = reduce_avg(losses, seqlen, dim=1)
    return tf.reduce_mean(losses)

@tf.function
def train_one_step(model, optimizer, x, y, seqlen):
    '''
    完成一步优化过程，可以参考之前做过的模型
    '''
    """单步训练流程（reduce_avg）
    参数:
        model: RNN模型实例
        optimizer: 优化器
        x: 输入序列 [batch_size, seq_len]
        y: 目标序列 [batch_size, seq_len]
        seqlen: 有效序列长度（已减1） [batch_size]
    返回:
        标量损失值
    """
    with tf.GradientTape() as tape:
        # 前向传播 [batch, seq_len, vocab_size]
        logits = model(x)
        
        # 计算掩码损失（使用reduce_avg）
        loss = compute_loss(logits, y, seqlen)
    
    # 梯度计算与参数更新
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    accuracy = 0.0
    for step, (x, y, seqlen) in enumerate(ds):
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy())

    return loss

# 训练优化过程

In [10]:
optimizer = optimizers.adam_v2.Adam(0.0005)
train_ds, word2id, id2word = poem_dataset()
model = myRNNModel(word2id)

for epoch in range(10):
    loss = train(epoch, model, optimizer, train_ds)

epoch 0 : loss 8.820669
epoch 1 : loss 6.565195
epoch 2 : loss 6.05506
epoch 3 : loss 5.8407664
epoch 4 : loss 5.62493
epoch 5 : loss 5.5387464
epoch 6 : loss 5.418851
epoch 7 : loss 5.4080544
epoch 8 : loss 5.25635
epoch 9 : loss 5.2379503


# 生成过程

In [24]:

import random

def generate_poem(begin_word, model, w2id, id2w, poem_type=random.choice([5,7])):
    state = model.rnncell.get_initial_state(batch_size=1, dtype=tf.float32)
    current_word = tf.constant([w2id[begin_word]], dtype=tf.int32)
    poem_lines = []
    current_line = [begin_word]
    
    # 忽略的符号列表（模型可能生成的无效字符）
    banned_tokens = ['，', '。', 'eos', 'UNK']
    
    for _ in range(4 * poem_type * 4):  # 放宽生成次数限制
        current_word, state = model.get_next_token(current_word, state)
        word = id2w.get(current_word.numpy()[0], 'UNK')
        
        # 关键逻辑：过滤无效符号
        if word in banned_tokens:
            continue  # 直接跳过，不加入诗句
        
        current_line.append(word)
        
        # 每句长度达标后添加标点
        if len(current_line) == poem_type:
            punctuation = '，' if len(poem_lines) < 3 else '。'
            poem_lines.append(''.join(current_line) + punctuation)
            current_line = []
        
        # 生成四句后强制终止
        if len(poem_lines) >= 4:
            break
    
    # 补全最后一句（如果未完成）
    if len(poem_lines) < 4 and current_line:
        current_line = current_line[:poem_type]  # 截断至规定长度
        punctuation = '。' if len(poem_lines) >= 3 else '，'
        poem_lines.append(''.join(current_line) + punctuation)
    
    # 确保最终为四句
    return ''.join(poem_lines[:4])

begin_words = ["日", "红", "山", "夜", "湖", "海", "月"]

for word in begin_words:
    poem = generate_poem(word, model, word2id, id2word, poem_type=random.choice([5,7]))  # 生成七言诗
    print(f"【{word}】开头生成的诗：\n{poem}\n")


【日】开头生成的诗：
日暮风来何处别，何处不相思得不，知君不知此日来，来无处处不见此。

【红】开头生成的诗：
红氲畔鹧蓉峦翡，蓉屿鳗悴啭琶悴，认琶悴认琶琶仞，悴曾鹉仞黏蓉仞。

【山】开头生成的诗：
山里不得春，风落来不得，无人间来不，得无人事不。

【夜】开头生成的诗：
夜暮风吹落，月不见春风，落月不相思，不知何处归。

【湖】开头生成的诗：
湖上月中春，一片云声落，一片云来不，可知不得一。

【海】开头生成的诗：
海阳山一枝花一，片云有山一月长，有山一月长有山，一月长有山一月。

【月】开头生成的诗：
月不得人间不人，不可知不得何人，间有一时何处处，不知何处是君人。

